In [ ]:
import os
import numpy as np
import xarray as xr
import zarr

import dask
import matplotlib.pyplot as plt

from dask.diagnostics import ProgressBar

In [ ]:
output_path = '/project/home/p200177/u101329/DE371_bis/MEPS_subdomain'

In [ ]:
#MEPS_path = '/project/home/p200177/DE_371/datasets/MEPS/aifs-meps-2.5km-2020-2023-1h-v2.zarr'
MEPS_path = '/mnt/tier2/project/p200177/u101834/DE371_bis/MEPS_subdomain/meps-2p5km-2020-2025-1h-v2.zarr_SMHI_subdomain.zarr'

In [ ]:
pds = xr.open_zarr(MEPS_path, zarr_format=2)

In [ ]:
#pds.attrs["variables_metadata"].items()

In [ ]:
print(pds)

In [ ]:
pds

In [ ]:
vars_keep = sorted(["10fg", "10u", "10v", "2t", "10si","tp","2d","z","vis","tcw","tcc","sp","mcc","lsm","lcc","hcc","fog","cbh",
'q_850',
't_850',
'u_850',
'v_850',
'w_850','msl'])

var_names = pds.attrs["variables"]

vidx = [var_names.index(v) for v in vars_keep]

In [ ]:
vars_keep

In [ ]:
len(vars_keep)

In [ ]:
ds_reduced = pds.isel(variable=vidx)

In [ ]:
ds_reduced = ds_reduced.drop_vars(
    ["minimum", "maximum", "mean", "sums", "squares", "stdev"],
    errors="ignore"
)

In [ ]:
ds_reduced.attrs["variables"] = vars_keep

# Optionally trim variables_metadata
if "variables_metadata" in ds_reduced.attrs:
    ds_reduced.attrs["variables_metadata"] = {
        k: v
        for k, v in ds_reduced.attrs["variables_metadata"].items()
        if k in vars_keep
    }

In [ ]:
dims = ("time", "ensemble", "cell")

data = ds_reduced["data"]

ds_reduced["minimum"] = data.min(dim=dims)
ds_reduced["maximum"] = data.max(dim=dims)
ds_reduced["sums"]    = data.sum(dim=dims)
ds_reduced["mean"]    = data.mean(dim=dims)
ds_reduced["squares"] = (data ** 2).sum(dim=dims)
ds_reduced["stdev"]   = data.std(dim=dims)
 
 

In [ ]:
output_path

In [ ]:
with ProgressBar():
    ds_reduced.to_zarr(
        f"{output_path}/meps-2p5km-2020-2025-1h-v2_subdomain_subvars.zarr",
        mode="w",
        zarr_format=2,
    )

In [ ]:
ds_red = xr.open_zarr(f"{output_path}/meps-2p5km-2020-2025-1h-v2_subdomain_subvars.zarr", zarr_format=2)

In [ ]:
print(ds_red)

In [ ]:
var_names = ds_red.attrs["variables"]
v_idx = var_names.index("10u")
print(v_idx)
field = ds_red["data"].isel(
    time=100,
    variable=v_idx,
    ensemble=0
)

In [ ]:
ds_red.attrs["variables"]

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    ds_red["longitudes"],
    ds_red["latitudes"],
    c=field,
    s=1
)
plt.colorbar(label="Wind speed")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Longtudes")
plt.show()

In [ ]:
d2 = np.array(field.data)

In [ ]:
d2.shape = (256,256)

In [ ]:
plt.imshow(d2)